# CSC2042S
## Assignment 3: Neural Networks
### Total marks: 30


#### Assignment goal: Optimisation of a Neural Network for Image Classification

This Jupyter Notebook will guide you through building a neural network, understanding the training process including backpropagation), and experimentally discovering how hyperparameters impact performance.

**Dataset: Fashion-MNIST** - A collection of 70,000 grayscale images of clothing items, perfect for testing your models.
https://www.kaggle.com/datasets/zalando-research/fashionmnist

## Part 1: Setup and Data Loading

* Import necessary libraries, such as ``torch``, ``torch.nn``, ``torchvision.transforms``, ``matplotlib.pyplot``, etc.

* Load the dataset and transform the dataset (convert to tensors, normalise). Load the Fashion_mNIST dataset into ``DataLoader`` objects for training and validation.

Data partitioning: Download the 60,000-image training dataset and split it into:
50,000 samples for training.
10,000 samples for validation.
Take into account Normalisation.


In [2]:
# Import libraries
from pathlib import Path
import numpy as np
import struct
import torch
from torch.utils.data import TensorDataset, DataLoader



data_dir = Path("data")
train_images_path = data_dir / "train-images-idx3-ubyte"
train_labels_path = data_dir / "train-labels-idx1-ubyte"
test_images_path = data_dir / "t10k-images-idx3-ubyte"
test_labels_path = data_dir / "t10k-labels-idx1-ubyte"

# Read data from the MNIST files
with open(train_images_path, 'rb') as f:
    magic_number, num_images, rows, cols = struct.unpack('>IIII', f.read(16))
    train_images = np.frombuffer(f.read(), dtype=np.uint8).reshape(num_images, rows, cols)

with open(train_labels_path, 'rb') as f:
    magic_number, num_labels = struct.unpack('>II', f.read(8))
    train_labels = np.frombuffer(f.read(), dtype=np.uint8)

with open(test_images_path, 'rb') as f:
    magic_number, num_images, rows, cols = struct.unpack('>IIII', f.read(16))
    test_images = np.frombuffer(f.read(), dtype=np.uint8).reshape(num_images, rows, cols)

with open(test_labels_path, 'rb') as f:
    magic_number, num_labels = struct.unpack('>II', f.read(8))
    test_labels = np.frombuffer(f.read(), dtype=np.uint8)

# Convert to tensors & scale to [0, 1] 
train_images = torch.tensor(train_images, dtype=torch.float32) / 255.0
train_images = train_images.unsqueeze(1)  
train_labels = torch.tensor(train_labels, dtype=torch.long)

test_images = torch.tensor(test_images, dtype=torch.float32) / 255.0
test_images = test_images.unsqueeze(1)   
test_labels = torch.tensor(test_labels, dtype=torch.long)

# Split indices 50,000 train / 10,000 validation
generator = torch.Generator().manual_seed(42)
perm = torch.randperm(len(train_images), generator=generator)
train_indices = perm[:50000]
val_indices   = perm[50000:]

# Compute mean and std using ONLY the 50,000 training images
train_mean = train_images[train_indices].mean()
train_std  = train_images[train_indices].std()

print("Training set normalisation statistics (from 50,000 training images):")
print(f"  Mean: {train_mean:.4f}")
print(f"  Std:  {train_std:.4f}")

# Normalize the training and test images
train_images = (train_images - train_mean) / train_std
test_images  = (test_images  - train_mean) / train_std

# Load data
train_dataset = TensorDataset(train_images[train_indices], train_labels[train_indices])
val_dataset   = TensorDataset(train_images[val_indices],   train_labels[val_indices])
test_dataset  = TensorDataset(test_images, test_labels)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# Runtime check
print(f"\nTraining set size:   {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size:       {len(test_dataset)}")

train_batch_imgs, train_batch_lbls = next(iter(train_loader))
val_batch_imgs, val_batch_lbls = next(iter(val_loader))

print(f"\nTrain batch images shape: {train_batch_imgs.shape}, labels shape: {train_batch_lbls.shape}")
print(f"Train batch pixel range:  [{train_batch_imgs.min():.2f}, {train_batch_imgs.max():.2f}]")
print(f"Train batch label range:  [{train_batch_lbls.min()}, {train_batch_lbls.max()}]")

print(f"\nVal batch images shape:   {val_batch_imgs.shape}, labels shape: {val_batch_lbls.shape}")
print(f"Val batch pixel range:    [{val_batch_imgs.min():.2f}, {val_batch_imgs.max():.2f}]")
print(f"Val batch label range:    [{val_batch_lbls.min()}, {val_batch_lbls.max()}]")



Training set normalisation statistics (from 50,000 training images):
  Mean: 0.2857
  Std:  0.3529

Training set size:   50000
Validation set size: 10000
Test set size:       10000

Train batch images shape: torch.Size([64, 1, 28, 28]), labels shape: torch.Size([64])
Train batch pixel range:  [-0.81, 2.02]
Train batch label range:  [0, 9]

Val batch images shape:   torch.Size([64, 1, 28, 28]), labels shape: torch.Size([64])
Val batch pixel range:    [-0.81, 2.02]
Val batch label range:    [0, 9]


## Part 2: Building and Training a Baseline Model [5 Marks]

Before we can optimise a model, we need a baseline to compare against. We will build a simple, standard neural network and train it with default settings.

**Define the Network Architecture**
Define a PyTorch class for your neural network that inherits from ``nn.Module``. Your model should be a simple Multi-Layer Perceptron (MLP) with the following architecture:

* Input Layer (a flattening layer).

* Hidden Layer 1: 128 neurons, ReLU activation.

* Hidden Layer 2: 64 neurons, ReLU activation.

* Output Layer: 10 neurons (one for each class).

**Training Loop**
Complete the provided ``train_model`` function. You will need to fill in the essential steps of the training process within the loop. This is where you will implement the logic for the forward pass, loss calculation, backpropagation, and weight updates.

**Train the Baseline Model**
Create an instance of your model, define a loss function, and an optimiser. Train your model for 10 epochs and record its final accuracy on the validation set. This is your baseline accuracy.

In [4]:
# Define Network Architecture [2 Marks]
import torch 
import torch.nn as nn
import torch.optim as optim

class BaselineMLP(nn.Module):
    def __init__(self):
        super(BaselineMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
      x = self.flatten(x)
      x = self.relu(self.fc1(x))
      x = self.relu(self.fc2(x))
      x = self.fc3(x)
      return x

In [5]:
# Updated Training and Evaluation skeleton [3 Marks]
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    """
    Trains a model and validates it each epoch.
    Returns: A dictionary containing lists of training loss, validation loss,
             and validation accuracy for each epoch.
    """
    # Data structure to store metrics
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    print("Starting Training...")
    for epoch in range(epochs):
        model.train() # Set model to training mode
        running_train_loss = 0.0
        for images, labels in train_loader:
            optimizer.zero_grad() # Zero the gradients
            outputs = model(images) # Forward pass
            loss = criterion(outputs, labels) # Compute loss
            loss.backward() # Backward pass
            optimizer.step() # Update weights
            running_train_loss += loss.item() * images.size(0)
            

        # Calculate and store average training loss for the epoch
        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        history['train_loss'].append(epoch_train_loss)

        # 2. Validation Phase
        model.eval() # Set model to evaluation mode
        running_val_loss = 0.0
        correct_predictions = 0
        total_predictions = 0

        with torch.no_grad(): # Disable gradient calculations
            for images, labels in val_loader:
                outputs = model(images) # Forward pass
                loss = criterion(outputs, labels) # Compute loss
                running_val_loss += loss.item() * images.size(0)

                _, predicted = torch.max(outputs, 1)
                correct_predictions += (predicted == labels).sum().item()
                total_predictions += labels.size(0)
   

        # Calculate and store average validation loss and accuracy
        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_val_acc = correct_predictions / total_predictions
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")

    print("Finished Training")
    return history

In [6]:
# Train the Baseline Model [3 Marks]
torch.manual_seed(42)  
model = BaselineMLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

baseline_history = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10)
print(f"\nBaseline validation accuracy: {baseline_history['val_acc'][-1]:.4f}")

Starting Training...
Epoch 1/10 - Train Loss: 0.4958, Val Loss: 0.4180, Val Acc: 0.8453
Epoch 2/10 - Train Loss: 0.3618, Val Loss: 0.3883, Val Acc: 0.8601
Epoch 3/10 - Train Loss: 0.3239, Val Loss: 0.3379, Val Acc: 0.8776
Epoch 4/10 - Train Loss: 0.2976, Val Loss: 0.3285, Val Acc: 0.8797
Epoch 5/10 - Train Loss: 0.2792, Val Loss: 0.3333, Val Acc: 0.8774
Epoch 6/10 - Train Loss: 0.2615, Val Loss: 0.3091, Val Acc: 0.8907
Epoch 7/10 - Train Loss: 0.2489, Val Loss: 0.3091, Val Acc: 0.8903
Epoch 8/10 - Train Loss: 0.2343, Val Loss: 0.3214, Val Acc: 0.8882
Epoch 9/10 - Train Loss: 0.2242, Val Loss: 0.3105, Val Acc: 0.8890
Epoch 10/10 - Train Loss: 0.2142, Val Loss: 0.3281, Val Acc: 0.8852
Finished Training

Baseline validation accuracy: 0.8852


## Part 3: Hyperparameter Optimisation Experiment [10 Marks]

Remember the definition of hyperparameters - they are external configuration variables set before training begins, controlling the model's structure and the learning process itself.
The goal is to find the set of the values that beats the baseline accuracy.

**Design and run your experiment**
Choose **three** from the following categories of hyperparameters to investigate:
* Learning Processes
* Regularisation/Architecture
  

Write a script (e.g., using nested loops) that systematically trains and evaluates a new model for each combination of the hyperparameters you have chosen. Store the final validation accuracy for each run in a data structure.

In [10]:
# Hyperparameter Optimisation Experiment [15 Marks]
import time

# Definition of hyperparameters config 
experiments = [
    # SGD runs
    {'optimizer_name': 'SGD',  'lr': 0.01,  'batch_size': 64},
    {'optimizer_name': 'SGD',  'lr': 0.01,  'batch_size': 128},
    {'optimizer_name': 'SGD',  'lr': 0.10,  'batch_size': 64},
    {'optimizer_name': 'SGD',  'lr': 0.10,  'batch_size': 128},
    # Adam runs
    {'optimizer_name': 'Adam', 'lr': 0.001, 'batch_size': 64},
    {'optimizer_name': 'Adam', 'lr': 0.001, 'batch_size': 128},
    {'optimizer_name': 'Adam', 'lr': 0.003, 'batch_size': 64},
    {'optimizer_name': 'Adam', 'lr': 0.003, 'batch_size': 128},
]

results = []
all_histories = []

best_val_acc = -1.0
best_model = None
best_history = None
best_config = None

criterion = nn.CrossEntropyLoss()
epochs = 10

print(f"Starting Hyperparameter Search: {len(experiments)} configurations, {epochs} epochs each...\n")
start_total_time = time.time()

for i, config in enumerate(experiments, 1):
    opt_name = config['optimizer_name']
    lr = config['lr']
    bs = config['batch_size']
    run_name = f"Run_{i}_{opt_name}_lr{lr}_bs{bs}"
    print(f"[{i}/{len(experiments)}] Running: {opt_name} | lr: {lr} | batch_size: {bs}")

    torch.manual_seed(42)
    model = BaselineMLP()

    # creating dataloaders for specific batch size
    exp_train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
    exp_val_loader   = DataLoader(val_dataset,   batch_size=bs, shuffle=False)

    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
    history = train_model(
        model = model, 
        train_loader = exp_train_loader,
        val_loader = exp_val_loader,
        criterion = criterion,
        optimizer = optimizer,
        epochs = epochs
    )
    
    finaal_train_loss = history['train_loss'][-1]
    final_val_loss = history['val_loss'][-1]
    final_val_acc = history['val_acc'][-1]
    
    record = {
        'Run': i,
        'Optimizer': opt_name,
        'Learning Rate': lr,
        'Batch Size': bs,
        'Final Train Loss': finaal_train_loss,
        'Final Val Loss': final_val_loss,
        'Final Val Acc': final_val_acc
    }
    results.append(record)
    all_histories.append(history)
    
    if final_val_acc > best_val_acc:
        best_val_acc = final_val_acc
        best_model = model
        best_history = history
        best_config = config
    

    
    print(f"--> Result: Val Acc = {final_val_acc * 100:.2f}%, Val Loss = {final_val_loss:.4f}\n")

total_duration = time.time() - start_total_time
print(f"Hyperparameter experiment completed in {total_duration / 60:.1f} minutes.")
print(f"Single Best Configuration: {best_config} with Validation Accuracy: {best_val_acc * 100:.2f}%")


Starting Hyperparameter Search: 8 configurations, 10 epochs each...

[1/8] Running: SGD | lr: 0.01 | batch_size: 64
Starting Training...
Epoch 1/10 - Train Loss: 0.9518, Val Loss: 0.6780, Val Acc: 0.7549
Epoch 2/10 - Train Loss: 0.5299, Val Loss: 0.5122, Val Acc: 0.8231
Epoch 3/10 - Train Loss: 0.4686, Val Loss: 0.4832, Val Acc: 0.8245
Epoch 4/10 - Train Loss: 0.4370, Val Loss: 0.4456, Val Acc: 0.8384
Epoch 5/10 - Train Loss: 0.4145, Val Loss: 0.4291, Val Acc: 0.8467
Epoch 6/10 - Train Loss: 0.3976, Val Loss: 0.4444, Val Acc: 0.8355
Epoch 7/10 - Train Loss: 0.3837, Val Loss: 0.4336, Val Acc: 0.8400
Epoch 8/10 - Train Loss: 0.3724, Val Loss: 0.3993, Val Acc: 0.8583
Epoch 9/10 - Train Loss: 0.3613, Val Loss: 0.4668, Val Acc: 0.8195
Epoch 10/10 - Train Loss: 0.3520, Val Loss: 0.3902, Val Acc: 0.8590
Finished Training
--> Result: Val Acc = 85.90%, Val Loss = 0.3902

[2/8] Running: SGD | lr: 0.01 | batch_size: 128
Starting Training...
Epoch 1/10 - Train Loss: 1.2584, Val Loss: 0.7395, Val A

## Part 4: Analysis and Report  [10 Marks]
In this final section, you will analyse the results of your experiment and report your findings.

* Present your experimental results in a well-formatted table (e.g., a Pandas DataFrame) showing the combination of hyperparameters and the final validation accuracy for each run. [2 Marks]
* Create at least two visualisations to help interpret your results: [4 Marks]
    * Hyperparameter Impact: Create a bar chart or heatmap that clearly compares the performance of your different hyperparameter combinations.
    * Loss Curves. Plot the training and validation loss curves (loss vs. epochs) for your baseline model and your single best-performing model on the same graph. This is crucial for comparing model convergence and overfitting.
    * Confusion Matrix. Generate and display a confusion matrix for your best model's predictions on the validation set. This will help you see which specific clothing items are being confused.
* Interpret your results. [4 Marks]
  * Answer the following questions concisely, using evidence from your table and plots to support your claims:
    * Best vs. Baseline: Which combination of hyperparameters produced the best model? How much did it improve upon the baseline accuracy?
    * Hyperparameter Influence: Which hyperparameter had the most significant impact on performance? Explain why you think certain values worked better than others.
    * Overfitting Analysis: By comparing the loss curves of your baseline and best models, analyze the degree of overfitting. Did your hyperparameter tuning help mitigate overfitting? How can you tell from the plot?
    * Error Analysis: Look at your confusion matrix. Identify two classes that the model frequently confuses. Propose a reason for this confusion by considering the visual characteristics of the items in those classes.